# Read gold through Horizon Catalog (PyIceberg)
Snowflake-managed Iceberg table `CDC_LAB.GOLD.TRANSACTION_ENRICHED`, read without a warehouse: Horizon (Iceberg REST) returns the metadata and short-lived S3 credentials; PyIceberg reads the Parquet files.

Set up by the notebook instance's start script: `horizon.py` and `horizon.env` next to this notebook, `pyiceberg` installed in the `python3` environment, the private key in Secrets Manager (read with the instance role).

In [ ]:
import os

for line in open("horizon.env"):  # SNOWFLAKE_ACCOUNT, HORIZON_SECRET_ID, AWS_DEFAULT_REGION
    key, _, value = line.strip().partition("=")
    os.environ[key] = value

import horizon

table = horizon.catalog().load_table(horizon.GOLD)
table.schema()

In [ ]:
rows = table.scan().to_arrow()
print(rows.num_rows, "rows")
rows.slice(0, 5).to_pandas()

In [ ]:
horizon.summarize(rows)

In [ ]:
rows.to_pandas().groupby(["CUSTOMER_TIER", "PRODUCT_TYPE"])["NET_AMOUNT"].agg(["count", "sum"]).head(10)